# 04 — Feature Engineering

**Goal:** Create new features with clear business rationale, verify they don't introduce leakage, and prepare the final feature set for modelling.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import load_raw_data, clean_data, separate_features_target, TARGET_COL
from src.feature_engineering import engineer_features

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)

In [ ]:
# Load and clean
df_raw = load_raw_data()
df = clean_data(df_raw)
print(f"Before feature engineering: {df.shape}")

## 4.1 Apply Feature Engineering

In [ ]:
df_eng = engineer_features(df)
print(f"\nAfter feature engineering: {df_eng.shape}")
print(f"\nNew columns: {[c for c in df_eng.columns if c not in df.columns]}")

## 4.2 Inspect New Features

In [ ]:
# Tenure groups
print("Tenure Group distribution:")
print(df_eng['tenure_group'].value_counts().sort_index())
print(f"\nChurn rate by tenure group:")
print(df_eng.groupby('tenure_group')[TARGET_COL].mean().sort_index())

In [ ]:
# Total services
print("Total services distribution:")
print(df_eng['total_services'].value_counts().sort_index())
print(f"\nChurn rate by total services:")
print(df_eng.groupby('total_services')[TARGET_COL].mean())

In [ ]:
# Average monthly charge
print(f"avg_monthly_charge — mean: {df_eng['avg_monthly_charge'].mean():.2f}, "
      f"std: {df_eng['avg_monthly_charge'].std():.2f}")
print(f"\ncharge_increase distribution:")
print(df_eng['charge_increase'].value_counts())
print(f"\nChurn rate with charge increase: {df_eng[df_eng['charge_increase']==1][TARGET_COL].mean():.3f}")
print(f"Churn rate without charge increase: {df_eng[df_eng['charge_increase']==0][TARGET_COL].mean():.3f}")

## 4.3 Visualise New Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn by tenure group
churn_by_tg = df_eng.groupby('tenure_group')[TARGET_COL].mean().sort_index()
churn_by_tg.plot(kind='bar', ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Churn Rate by Tenure Group', fontweight='bold')
axes[0].set_ylabel('Churn Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Churn by total services
churn_by_svc = df_eng.groupby('total_services')[TARGET_COL].mean()
churn_by_svc.plot(kind='bar', ax=axes[1], color='#e67e22', edgecolor='white')
axes[1].set_title('Churn Rate by Total Services', fontweight='bold')
axes[1].set_ylabel('Churn Rate')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Churn by charge increase
churn_by_ci = df_eng.groupby('charge_increase')[TARGET_COL].mean()
churn_by_ci.index = ['No Increase', 'Increase']
churn_by_ci.plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[2].set_title('Churn Rate by Charge Increase', fontweight='bold')
axes[2].set_ylabel('Churn Rate')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
fig.savefig('../reports/figures/engineered_features.png', dpi=150)
plt.show()

## 4.4 Leakage Verification

| Feature | Uses Target? | Uses Future Info? | Verdict |
|---|---|---|---|
| tenure_group | No | No | ✅ Safe |
| total_services | No | No | ✅ Safe |
| avg_monthly_charge | No | No (uses historical TotalCharges/tenure) | ✅ Safe |
| charge_increase | No | No (compares current vs historical avg) | ✅ Safe |
| has_internet | No | No | ✅ Safe |
| has_phone_and_internet | No | No | ✅ Safe |

All engineered features are derived from information available at prediction time.